In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Helper to build compact summary for a dataframe with timestamp and cmdb_id
def compact_summary(df, ts_col="timestamp", cmdb_col="cmdb_id", top_n_cmdb=20):
    rows = len(df)
    ts_min = pd.to_datetime(df[ts_col].min(), unit="s", utc=True) if rows > 0 else pd.NaT
    ts_max = pd.to_datetime(df[ts_col].max(), unit="s", utc=True) if rows > 0 else pd.NaT
    top_cmdb_series = df[cmdb_col].value_counts().head(top_n_cmdb)
    # represent top cmdbs as a short dataframe (value_counts) and as a compact comma-separated string
    top_cmdb_list = top_cmdb_series.index.tolist()
    top_cmdb_str = ", ".join(map(str, top_cmdb_list))
    summary_df = pd.DataFrame([{
        "rows": rows,
        "ts_min_utc": ts_min,
        "ts_max_utc": ts_max,
        "top_cmdb_ids (up to {})".format(top_n_cmdb): top_cmdb_str
    }])
    # also return the value_counts series for potential inspection (kept compact)
    return summary_df, top_cmdb_series

# Build summaries
metric_summary, metric_top_cmdb = compact_summary(metric, ts_col="timestamp", cmdb_col="cmdb_id", top_n_cmdb=20)
trace_summary, trace_top_cmdb = compact_summary(trace, ts_col="timestamp", cmdb_col="cmdb_id", top_n_cmdb=20)
log_summary, log_top_cmdb = compact_summary(log, ts_col="timestamp", cmdb_col="cmdb_id", top_n_cmdb=20)
error_logs_summary, error_logs_top_cmdb = compact_summary(error_logs, ts_col="timestamp", cmdb_col="cmdb_id", top_n_cmdb=20)

# Top KPI/feature/name counts (top 20) for metric/trace/log as requested
metric_kpi_top20 = metric["kpi_name"].value_counts().head(20).reset_index().rename(columns={"index":"kpi_name","kpi_name":"count"})
trace_top20 = trace["trace_name"].value_counts().head(20).reset_index().rename(columns={"index":"trace_name","trace_name":"count"})
log_top20 = log["log_name"].value_counts().head(20).reset_index().rename(columns={"index":"log_name","log_name":"count"})

# Keep final displayed output compact as required
metric_summary, metric_kpi_top20, trace_summary, trace_top20, log_summary, log_top20, error_logs_summary

```
Out[1]:
```
```python
import pandas as pd

metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

def stats(df, ts_col="timestamp", id_col="cmdb_id"):
    rows = len(df)
    if rows:
        ts_min = pd.to_datetime(df[ts_col].min(), unit="s", utc=True)
        ts_max = pd.to_datetime(df[ts_col].max(), unit="s", utc=True)
        top_ids = df[id_col].value_counts().head(20).index.tolist()
    else:
        ts_min = pd.NaT
        ts_max = pd.NaT
        top_ids = []
    return rows, ts_min, ts_max, top_ids

m_rows, m_min, m_max, m_ids = stats(metric)
t_rows, t_min, t_max, t_ids = stats(trace)
l_rows, l_min, l_max, l_ids = stats(log)
e_rows, e_min, e_max, e_ids = stats(error_logs)

m_kpis = metric["kpi_name"].value_counts().head(20)
t_names = trace["trace_name"].value_counts().head(20)
l_names = log["log_name"].value_counts().head(20)

summary = (
    f"Summary of loaded telemetry:\n\n"
    f"metric.csv: {m_rows} rows; time range {m_min} to {m_max} (UTC). "
    f"Top cmdb_ids (up to 20): {', '.join(map(str, m_ids[:20]))}.\n"
    f"Top metric kpi_names: " + ", ".join([f\"{k} ({v})\" for k,v in m_kpis.items()]) + ".\n\n"
    f"trace.csv: {t_rows} rows; time range {t_min} to {t_max} (UTC). "
    f"Top cmdb_ids (up to 20): {', '.join(map(str, t_ids[:20]))}.\n"
    f"Top trace_names: " + ", ".join([f\"{k} ({v})\" for k,v in t_names.items()]) + ".\n\n"
    f"log.csv: {l_rows} rows; time range {l_min} to {l_max} (UTC). "
    f"Top cmdb_ids (up to 20): {', '.join(map(str, l_ids[:20]))}.\n"
    f"Top log_names: " + ", ".join([f\"{k} ({v})\" for k,v in l_names.items()]) + ".\n\n"
    f"error_logs.csv: {e_rows} rows; time range {e_min} to {e_max} (UTC). No top cmdb_ids."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   rows                ts_min_utc                ts_max_utc                            top_cmdb_ids (up to 20)
0  1710 2024-01-20 15:33:00+00:00 2024-01-20 15:57:00+00:00  adservice, cartservice, emailservice, recommen...,         count  count
0         cpu    275
1      socket    275
2         mem    275
3  latency-90    250
4  latency-50    250
5    workload    250
6      diskio    135,    rows                ts_min_utc                ts_max_utc                            top_cmdb_ids (up to 20)
0  3800 2024-01-20 15:33:00+00:00 2024-01-20 15:57:00+00:00  checkoutservice, frontendservice, root, recomm...,                                            count  count
0                  trace.from_root.duration_mean    175
1                   trace.from_root.duration_p95    175
2                     trace.from_root.error_rate    175
3                      trace.from_root.row_count    175
4           trace.from_checkoutservice.row_count    125
5          trace.from_checkoutservice.error_rate    125
6        trace.from_checkoutservice.duration_p95    125
7       trace.from_checkoutservice.duration_mean    125
8           trace.from_frontendservice.row_count    125
9          trace.from_frontendservice.error_rate    125
10       trace.from_frontendservice.duration_p95    125
11      trace.from_frontendservice.duration_mean    125
12      trace.to_productcatalogservice.row_count    100
13     trace.to_productcatalogservice.error_rate    100
14   trace.to_productcatalogservice.duration_p95    100
15  trace.to_productcatalogservice.duration_mean    100
16        trace.to_checkoutservice.duration_mean     75
17         trace.to_checkoutservice.duration_p95     75
18           trace.to_checkoutservice.error_rate     75
19            trace.to_checkoutservice.row_count     75,    rows                ts_min_utc                ts_max_utc                            top_cmdb_ids (up to 20)
0   460 2024-01-20 15:33:00+00:00 2024-01-20 15:57:00+00:00  adservice, cartservice, checkoutservice, curre...,              count  count
0  log.error_count    230
1    log.row_count    230,    rows ts_min_utc ts_max_utc top_cmdb_ids (up to 20)
0     0        NaT        NaT                        )```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing metric variable if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# ensure timestamp parsed to UTC datetimes (rule 10)
if 'ts_utc' not in metric.columns:
    metric['ts_utc'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (before any filtering)
group_keys = ['cmdb_id', 'kpi_name']
p95 = metric.groupby(group_keys)['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95_threshold'})

# 2) Filter metric.csv to the incident window (UTC)
start = pd.Timestamp("2024-01-20 15:30:28", tz="UTC")
end = pd.Timestamp("2024-01-20 16:00:28", tz="UTC")
in_window = metric[(metric['ts_utc'] >= start) & (metric['ts_utc'] <= end)].copy()

# If no rows in window, produce an empty DataFrame with requested columns
if in_window.empty:
    result_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95_threshold','total_points_in_window',
        'anomaly_count','earliest_anomaly_utc','max_value_in_window'
    ])
else:
    # attach global p95 thresholds to rows in the window
    in_window = in_window.merge(p95, on=group_keys, how='left')

    # mark anomalies within window (value >= global p95 threshold)
    in_window['is_anom'] = in_window['value'] >= in_window['p95_threshold']

    # aggregates per (cmdb_id, kpi_name)
    counts = in_window.groupby(group_keys)['value'].size().reset_index(name='total_points_in_window')
    anomaly_counts = in_window.groupby(group_keys)['is_anom'].sum().reset_index(name='anomaly_count')
    max_vals = in_window.groupby(group_keys)['value'].max().reset_index(name='max_value_in_window')
    earliest_anoms = in_window[in_window['is_anom']].groupby(group_keys)['ts_utc'].min().reset_index(name='earliest_anomaly_utc')

    # merge aggregates and include p95_threshold
    result = counts.merge(anomaly_counts, on=group_keys)
    result = result.merge(earliest_anoms, on=group_keys, how='left')
    result = result.merge(max_vals, on=group_keys)
    result = result.merge(p95, on=group_keys)  # include global p95

    # reorder columns as requested
    result = result[['cmdb_id','kpi_name','p95_threshold','total_points_in_window',
                     'anomaly_count','earliest_anomaly_utc','max_value_in_window']]

    # 4) sort by anomaly_count desc and limit to top 20
    result_top20 = result.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Display compact result (up to 20 rows)
result_top20

```
Out[2]:
```
```python
# Summarize the previously computed result_top20 (compact plain-English summary)
try:
    df = result_top20
except NameError:
    # fallback: load result if not present (shouldn't be needed in the interactive session)
    import pandas as pd
    metric = pd.read_csv("metric.csv")
    metric['ts_utc'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
    start = pd.Timestamp("2024-01-20 15:30:28", tz="UTC")
    end = pd.Timestamp("2024-01-20 16:00:28", tz="UTC")
    p95 = metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_threshold'})
    in_window = metric[(metric['ts_utc'] >= start) & (metric['ts_utc'] <= end)].merge(p95, on=['cmdb_id','kpi_name'], how='left')
    in_window['is_anom'] = in_window['value'] >= in_window['p95_threshold']
    counts = in_window.groupby(['cmdb_id','kpi_name'])['value'].size().reset_index(name='total_points_in_window')
    anomaly_counts = in_window.groupby(['cmdb_id','kpi_name'])['is_anom'].sum().reset_index(name='anomaly_count')
    max_vals = in_window.groupby(['cmdb_id','kpi_name'])['value'].max().reset_index(name='max_value_in_window')
    earliest_anoms = in_window[in_window['is_anom']].groupby(['cmdb_id','kpi_name'])['ts_utc'].min().reset_index(name='earliest_anomaly_utc')
    df = counts.merge(anomaly_counts, on=['cmdb_id','kpi_name']).merge(earliest_anoms, on=['cmdb_id','kpi_name'], how='left').merge(max_vals, on=['cmdb_id','kpi_name']).merge(p95, on=['cmdb_id','kpi_name'])
    df = df[['cmdb_id','kpi_name','p95_threshold','total_points_in_window','anomaly_count','earliest_anomaly_utc','max_value_in_window']].sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Build a plain-English summary string from the top rows
window_start = "2024-01-20 15:30:28 UTC"
window_end = "2024-01-20 16:00:28 UTC"

# Top entries listing
top_lines = []
for i, row in df.iterrows():
    cmk = f"{row['cmdb_id']}.{row['kpi_name']}"
    anom = int(row['anomaly_count'])
    p95 = row['p95_threshold']
    first = row['earliest_anomaly_utc']
    first_str = str(first) if pd.notna(first) else "None"
    top_lines.append(f"{cmk}: {anom} anomalies (p95={p95}); first at {first_str}; max={row['max_value_in_window']}")

# Compose concise summary
summary = (
    f"Incident window: {window_start} to {window_end}.\n\n"
    f"Key findings (top anomalous metrics in the window):\n"
    f"- The most frequent anomalies are on the 'socket' KPI across many services. Several services show 25/25 points flagged as anomalies, e.g. currencyservice.socket, cartservice.socket, adservice.socket, redis.socket, productcatalogservice.socket, paymentservice.socket.\n"
    f"- Several services show 'diskio' with p95=0 and 25 anomalies (e.g. emailservice.diskio, cartservice.diskio, recommendationservice.diskio) indicating values met or exceeded their global p95 (which is 0).\n"
    f"- checkoutservice.socket and frontend.socket also have notable anomaly counts (12 and 8 respectively).\n"
    f"- adservice exhibits a few other anomalies (mem, cpu, latency-50, latency-90) but at much lower counts (2 each).\n\n"
    f"Top rows (compact):\n" + "\n".join(top_lines[:20]) + "\n\n"
    f"Interpretation: socket-related metrics dominate the anomalies in this window across multiple services, with diskio (p95=0) also flagged repeatedly. Checkout and frontend show partial elevated socket anomalies; adservice has a small number of memory/cpu/latency anomalies."
)

# Display the plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  p95_threshold  total_points_in_window  anomaly_count      earliest_anomaly_utc  max_value_in_window
0         currencyservice      socket   4.000000e+00                      25             25 2024-01-20 15:33:00+00:00         4.000000e+00
1            emailservice      diskio   0.000000e+00                      25             25 2024-01-20 15:33:00+00:00         0.000000e+00
2             cartservice      socket   5.000000e+00                      25             25 2024-01-20 15:33:00+00:00         5.000000e+00
3             cartservice      diskio   0.000000e+00                      25             25 2024-01-20 15:33:00+00:00         0.000000e+00
4               adservice      socket   4.000000e+00                      25             25 2024-01-20 15:33:00+00:00         4.000000e+00
5   recommendationservice      diskio   0.000000e+00                      25             25 2024-01-20 15:33:00+00:00         0.000000e+00
6   recommendationservice      socket   4.000000e+00                      25             25 2024-01-20 15:33:00+00:00         4.000000e+00
7         shippingservice      socket   2.000000e+00                      25             25 2024-01-20 15:33:00+00:00         2.000000e+00
8                   redis      socket   4.000000e+00                      25             25 2024-01-20 15:33:00+00:00         4.000000e+00
9   productcatalogservice      socket   4.000000e+00                      25             25 2024-01-20 15:33:00+00:00         4.000000e+00
10           emailservice      socket   3.000000e+00                      25             25 2024-01-20 15:33:00+00:00         3.000000e+00
11         paymentservice      socket   3.000000e+00                      25             25 2024-01-20 15:33:00+00:00         3.000000e+00
12        checkoutservice      socket   1.200000e+01                      25             12 2024-01-20 15:46:00+00:00         1.200000e+01
13               frontend      socket   2.400000e+01                      25              8 2024-01-20 15:50:00+00:00         2.400000e+01
14              adservice         mem   1.039254e+08                      25              2 2024-01-20 15:56:00+00:00         1.039973e+08
15              adservice  latency-50   3.105612e-03                      25              2 2024-01-20 15:33:00+00:00         3.376035e-03
16              adservice      diskio   2.510844e+05                      25              2 2024-01-20 15:49:00+00:00         2.778133e+05
17              adservice         cpu   3.431066e+00                      25              2 2024-01-20 15:33:00+00:00         3.584209e+00
18              adservice  latency-90   4.812523e-03                      25              2 2024-01-20 15:33:00+00:00         6.939796e-03
19            cartservice         mem   4.349466e+07                      25              2 2024-01-20 15:49:00+00:00         4.377873e+07```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse existing trace variable if available, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# ensure UTC timestamp column
if 'ts_utc' not in trace.columns:
    trace['ts_utc'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Global P95 per (cmdb_id, trace_name) computed on the full series
group_keys = ['cmdb_id', 'trace_name']
trace_p95 = trace.groupby(group_keys)['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95_threshold'})

# 2) Filter to incident window (UTC)
start = pd.Timestamp("2024-01-20 15:30:28", tz="UTC")
end = pd.Timestamp("2024-01-20 16:00:28", tz="UTC")
trace_window = trace[(trace['ts_utc'] >= start) & (trace['ts_utc'] <= end)].copy()

# Prepare empty result if nothing in window
cols = ['cmdb_id','trace_name','p95_threshold','total_points_in_window','anomaly_count','earliest_anomaly_utc','max_value_in_window']
if trace_window.empty:
    trace_result_top20 = pd.DataFrame(columns=cols)
else:
    # attach global p95 to window rows
    trace_window = trace_window.merge(trace_p95, on=group_keys, how='left')
    # mark anomalies (value >= global p95)
    trace_window['is_anom'] = trace_window['value'] >= trace_window['p95_threshold']
    # aggregations
    total_pts = trace_window.groupby(group_keys)['value'].size().reset_index(name='total_points_in_window')
    anom_cnt = trace_window.groupby(group_keys)['is_anom'].sum().reset_index(name='anomaly_count')
    max_val = trace_window.groupby(group_keys)['value'].max().reset_index(name='max_value_in_window')
    earliest = trace_window[trace_window['is_anom']].groupby(group_keys)['ts_utc'].min().reset_index(name='earliest_anomaly_utc')
    # merge aggregates and include p95
    trace_result = total_pts.merge(anom_cnt, on=group_keys)
    trace_result = trace_result.merge(earliest, on=group_keys, how='left')
    trace_result = trace_result.merge(max_val, on=group_keys)
    trace_result = trace_result.merge(trace_p95, on=group_keys)
    # select and order columns
    trace_result = trace_result[['cmdb_id','trace_name','p95_threshold','total_points_in_window',
                                 'anomaly_count','earliest_anomaly_utc','max_value_in_window']]
    # 4) sort by anomaly_count desc and limit to top 20
    trace_result_top20 = trace_result.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Display compact result
trace_result_top20

```
Out[3]:
```
```python
summary = (
    "Incident window: 2024-01-20 15:30:28 to 2024-01-20 16:00:28 UTC.\n\n"
    "Findings (trace.csv):\n"
    "- The top 20 rows are all trace '*.error_rate' metrics across many services (checkoutservice, frontendservice, paymentservice, recommendationservice, root, etc.).\n"
    "- For each listed series the global P95 = 0.0, total points in window = 25, anomaly_count = 25, earliest anomaly = 2024-01-20 15:33:00+00:00, and max value in window = 0.0.\n\n"
    "Interpretation:\n"
    "- Because the global P95 threshold is 0.0, every value in the window (including zeros) meets or exceeds the threshold, so anomaly_count is saturated (25/25) for many series.\n"
    "- This makes these 'anomalies' non-actionable as reported; the thresholding logic (value >= p95 with p95==0) produces false-positive saturation. Recompute thresholds or use a stricter rule (e.g., > p95 or a nonzero baseline) before treating these as true error-rate incidents."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  p95_threshold  total_points_in_window  anomaly_count      earliest_anomaly_utc  max_value_in_window
0         checkoutservice      trace.from_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
1         checkoutservice      trace.from_frontendservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
2         checkoutservice        trace.to_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
3         checkoutservice                 trace.from_root.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
4         checkoutservice        trace.to_currencyservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
5         checkoutservice           trace.to_emailservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
6                    root        trace.to_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
7   recommendationservice  trace.to_recommendationservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
8         frontendservice                 trace.from_root.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
9         frontendservice        trace.to_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
10        frontendservice        trace.to_frontendservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
11        frontendservice        trace.to_currencyservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
12         paymentservice      trace.from_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
13         paymentservice                 trace.from_root.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
14        frontendservice  trace.to_recommendationservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
15        frontendservice  trace.to_productcatalogservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
16        currencyservice                 trace.from_root.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
17           emailservice      trace.from_checkoutservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
18        frontendservice      trace.from_frontendservice.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0
19           emailservice                 trace.from_root.error_rate            0.0                      25             25 2024-01-20 15:33:00+00:00                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# reuse existing trace variable if present, otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")

# ensure UTC timestamp column
if 'ts_utc' not in trace.columns:
    trace['ts_utc'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Restrict to trace_name values that contain 'duration'
trace_duration_full = trace[trace['trace_name'].str.contains('duration', na=False)].copy()

# 2) Compute global P95 per (cmdb_id, trace_name) on the full duration series (before filtering window)
group_keys = ['cmdb_id', 'trace_name']
p95 = trace_duration_full.groupby(group_keys)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_threshold'})

# 3) Filter rows within the incident window (UTC)
start = pd.Timestamp("2024-01-20 15:30:28", tz="UTC")
end = pd.Timestamp("2024-01-20 16:00:28", tz="UTC")
trace_duration_window = trace_duration_full[(trace_duration_full['ts_utc'] >= start) & (trace_duration_full['ts_utc'] <= end)].copy()

# 4) Aggregate per (cmdb_id, trace_name)
cols = ['cmdb_id','trace_name','p95_threshold','total_points_in_window','anomaly_count',
        'earliest_anomaly_utc','max_value_in_window','ratio']

if trace_duration_window.empty:
    trace_duration_top20 = pd.DataFrame(columns=cols)
else:
    # attach global p95 thresholds
    trace_duration_window = trace_duration_window.merge(p95, on=group_keys, how='left')
    # mark anomalies (value >= global p95)
    trace_duration_window['is_anom'] = trace_duration_window['value'] >= trace_duration_window['p95_threshold']
    # aggregations
    total_pts = trace_duration_window.groupby(group_keys)['value'].size().reset_index(name='total_points_in_window')
    anom_cnt = trace_duration_window.groupby(group_keys)['is_anom'].sum().reset_index(name='anomaly_count')
    max_val = trace_duration_window.groupby(group_keys)['value'].max().reset_index(name='max_value_in_window')
    earliest = trace_duration_window[trace_duration_window['is_anom']].groupby(group_keys)['ts_utc'].min().reset_index(name='earliest_anomaly_utc')
    # merge aggregates and include p95
    result = total_pts.merge(anom_cnt, on=group_keys)
    result = result.merge(earliest, on=group_keys, how='left')
    result = result.merge(max_val, on=group_keys)
    result = result.merge(p95, on=group_keys)
    # compute ratio (NaN when p95_threshold == 0)
    result['ratio'] = result.apply(lambda r: (r['max_value_in_window'] / r['p95_threshold']) 
                                   if (pd.notna(r['p95_threshold']) and r['p95_threshold'] != 0) else np.nan, axis=1)
    # keep only rows with total_points_in_window > 0 (should already be true)
    result = result[result['total_points_in_window'] > 0]
    # select and order columns
    trace_duration_top20 = result[['cmdb_id','trace_name','p95_threshold','total_points_in_window',
                                  'anomaly_count','earliest_anomaly_utc','max_value_in_window','ratio']] \
                         .sort_values(by='ratio', ascending=False, na_position='last') \
                         .head(20).reset_index(drop=True)

# Display compact result variable
trace_duration_top20

```
Out[4]:
```
```python
# Build a concise plain-English summary from trace_duration_top20
import pandas as pd

df = trace_duration_top20.copy()

window = "2024-01-20 15:30:28 to 2024-01-20 16:00:28 UTC"

top_lines = []
for _, r in df.head(10).iterrows():
    name = f"{r['cmdb_id']}.{r['trace_name']}"
    ratio = r['ratio']
    first = r['earliest_anomaly_utc']
    maxv = r['max_value_in_window']
    p95 = r['p95_threshold']
    top_lines.append(f"- {name}: ratio={ratio:.2f}, max={maxv}, p95={p95}, first_anomaly={first}")

summary = (
    f"Incident window: {window}.\n\n"
    f"Key findings for 'duration' traces:\n"
    f"1) The largest relative duration spikes are between root and currencyservice:\n"
    f"   - root.trace.to_currencyservice.duration_p95 and currencyservice.trace.from_root.duration_p95 have the highest ratio (~58.7x), with the first anomalies at 2024-01-20 15:54:00 UTC.\n"
    f"   - Corresponding duration_mean series for currencyservice/root also show large ratios (~39.5x).\n"
    f"2) Smaller but notable spikes occur around checkout/payment paths (ratios ~1.7–1.9), with first anomalies at ~15:39 UTC.\n"
    f"3) Most listed series have only 2 anomaly points in the window (anomaly_count = 2) while total_points_in_window = 25, so these are short-lived spikes.\n\n"
    f"Top examples (up to 10):\n" + "\n".join(top_lines) + "\n\n"
    f"Interpretation: The most significant outliers are short-duration latency spikes involving currencyservice (at ~15:54 UTC). Investigate currencyservice interactions at that time; also check checkout↔payment traces around 15:39 UTC."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  p95_threshold  total_points_in_window  anomaly_count      earliest_anomaly_utc  max_value_in_window      ratio
0                    root         trace.to_currencyservice.duration_p95       0.000497                      25              2 2024-01-20 15:54:00+00:00             0.029140  58.665821
1         currencyservice                  trace.from_root.duration_p95       0.000497                      25              2 2024-01-20 15:54:00+00:00             0.029140  58.665821
2         currencyservice                 trace.from_root.duration_mean       0.000138                      25              2 2024-01-20 15:54:00+00:00             0.005439  39.520465
3                    root        trace.to_currencyservice.duration_mean       0.000138                      25              2 2024-01-20 15:54:00+00:00             0.005439  39.520465
4          paymentservice      trace.from_checkoutservice.duration_mean       0.000506                      25              2 2024-01-20 15:39:00+00:00             0.000968   1.911735
5         checkoutservice         trace.to_paymentservice.duration_mean       0.000506                      25              2 2024-01-20 15:39:00+00:00             0.000968   1.911735
6         checkoutservice          trace.to_paymentservice.duration_p95       0.000959                      25              2 2024-01-20 15:39:00+00:00             0.001694   1.765810
7          paymentservice       trace.from_checkoutservice.duration_p95       0.000959                      25              2 2024-01-20 15:39:00+00:00             0.001694   1.765810
8                    root            trace.to_emailservice.duration_p95       0.000157                      25              2 2024-01-20 15:33:00+00:00             0.000246   1.564468
9            emailservice                  trace.from_root.duration_p95       0.000157                      25              2 2024-01-20 15:33:00+00:00             0.000246   1.564468
10        checkoutservice        trace.to_currencyservice.duration_mean       0.001688                      25              2 2024-01-20 15:39:00+00:00             0.002579   1.527951
11        currencyservice      trace.from_checkoutservice.duration_mean       0.001688                      25              2 2024-01-20 15:39:00+00:00             0.002579   1.527951
12        checkoutservice      trace.from_checkoutservice.duration_mean       0.017262                      25              2 2024-01-20 15:48:00+00:00             0.024932   1.444286
13        checkoutservice        trace.to_checkoutservice.duration_mean       0.017262                      25              2 2024-01-20 15:48:00+00:00             0.024932   1.444286
14                   root  trace.to_recommendationservice.duration_mean       0.000066                      25              2 2024-01-20 15:36:00+00:00             0.000091   1.375016
15  recommendationservice                 trace.from_root.duration_mean       0.000066                      25              2 2024-01-20 15:36:00+00:00             0.000091   1.375016
16                   root          trace.to_paymentservice.duration_p95       0.000442                      25              2 2024-01-20 15:37:00+00:00             0.000564   1.274787
17         paymentservice                  trace.from_root.duration_p95       0.000442                      25              2 2024-01-20 15:37:00+00:00             0.000564   1.274787
18  recommendationservice                  trace.from_root.duration_p95       0.000130                      25              2 2024-01-20 15:36:00+00:00             0.000165   1.265541
19                   root   trace.to_recommendationservice.duration_p95       0.000130                      25              2 2024-01-20 15:36:00+00:00             0.000165   1.265541```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# reuse existing 'log' variable if available, otherwise load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# Candidate services
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# Ensure UTC timestamp column
if 'ts_utc' not in log.columns:
    log['ts_utc'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# 1) Restrict to candidate services for analysis (thresholds computed per series among these services)
log_cand = log[log['cmdb_id'].isin(candidates)].copy()

# 2) Compute global P95 per (cmdb_id, log_name) using the full available series (no window filtering)
group_keys = ['cmdb_id', 'log_name']
p95 = log_cand.groupby(group_keys)['value'].quantile(0.95).reset_index().rename(columns={'value':'p95_threshold'})

# Also compute global P15 for log.row_count series (for step 5)
rowcount_p15 = (
    log_cand[log_cand['log_name'] == 'log.row_count']
    .groupby(['cmdb_id','log_name'])['value']
    .quantile(0.15)
    .reset_index()
    .rename(columns={'value':'p15_threshold'})
)

# 3) Filter rows within the incident window (UTC)
start = pd.Timestamp("2024-01-20 15:30:28", tz="UTC")
end = pd.Timestamp("2024-01-20 16:00:28", tz="UTC")
log_window = log_cand[(log_cand['ts_utc'] >= start) & (log_cand['ts_utc'] <= end)].copy()

# Prepare output columns
cols = ['cmdb_id','log_name','p95_threshold','total_points_in_window','anomaly_count',
        'earliest_anomaly_utc','max_value_in_window','below_p15_count','earliest_below_p15_utc']

if log_window.empty:
    log_result_top20 = pd.DataFrame(columns=cols)
else:
    # 4) Merge p95 into window rows and mark anomalies (value >= p95_threshold)
    log_window = log_window.merge(p95, on=group_keys, how='left')
    log_window['is_anom'] = log_window['value'] >= log_window['p95_threshold']

    # Aggregations: total points, anomaly_count, max value, earliest anomaly
    total_pts = log_window.groupby(group_keys)['value'].size().reset_index(name='total_points_in_window')
    anom_cnt = log_window.groupby(group_keys)['is_anom'].sum().reset_index(name='anomaly_count')
    max_val = log_window.groupby(group_keys)['value'].max().reset_index(name='max_value_in_window')
    earliest_anom = (log_window[log_window['is_anom']]
                     .groupby(group_keys)['ts_utc'].min()
                     .reset_index(name='earliest_anomaly_utc'))

    # 5) For 'log.row_count' series compute in-window values below global P15
    # Merge p15 thresholds into window rows (only matches where log_name == 'log.row_count')
    log_window = log_window.merge(rowcount_p15, on=group_keys, how='left')
    # Flag below p15 only for log.row_count rows (p15_threshold not null)
    log_window['below_p15'] = np.where(
        (log_window['log_name'] == 'log.row_count') & (pd.notna(log_window['p15_threshold'])) & (log_window['value'] < log_window['p15_threshold']),
        True, False
    )
    below_counts = (log_window[log_window['below_p15']]
                    .groupby(group_keys)['value']
                    .size()
                    .reset_index(name='below_p15_count'))
    earliest_below = (log_window[log_window['below_p15']]
                      .groupby(group_keys)['ts_utc']
                      .min()
                      .reset_index(name='earliest_below_p15_utc'))

    # Merge all aggregates together
    result = total_pts.merge(anom_cnt, on=group_keys)
    result = result.merge(earliest_anom, on=group_keys, how='left')
    result = result.merge(max_val, on=group_keys)
    result = result.merge(p95, on=group_keys, how='left')
    # merge below-p15 aggregates (may be empty for many series)
    result = result.merge(below_counts, on=group_keys, how='left')
    result = result.merge(earliest_below, on=group_keys, how='left')

    # Fill NaN below_p15_count with 0
    if 'below_p15_count' in result.columns:
        result['below_p15_count'] = result['below_p15_count'].fillna(0).astype(int)
    else:
        result['below_p15_count'] = 0
    # Ensure earliest_below_p15_utc column exists
    if 'earliest_below_p15_utc' not in result.columns:
        result['earliest_below_p15_utc'] = pd.NaT

    # Select requested columns and sort by anomaly_count desc, limit to top 20
    result = result[['cmdb_id','log_name','p95_threshold','total_points_in_window',
                     'anomaly_count','earliest_anomaly_utc','max_value_in_window',
                     'below_p15_count','earliest_below_p15_utc']]

    log_result_top20 = result.sort_values(by='anomaly_count', ascending=False).head(20).reset_index(drop=True)

# Display compact result
log_result_top20

```
Out[5]:
```
```python
# Build a plain-English summary from log_result_top20
try:
    df = log_result_top20.copy()
except NameError:
    import pandas as pd
    df = pd.DataFrame([
        # fallback empty
    ])

# Compose summary
lines = []
lines.append("Incident window: 2024-01-20 15:30:28 to 2024-01-20 16:00:28 UTC.")
lines.append("")
# Error count findings
err_df = df[df['log_name'] == 'log.error_count']
if not err_df.empty:
    svc_list = err_df['cmdb_id'].tolist()
    lines.append("log.error_count:")
    lines.append(f"- Services with saturated 'error_count' anomalies: {', '.join(svc_list)}.")
    lines.append("- For these series the global P95 = 0.0, total points in window = 25, anomaly_count = 25, earliest anomaly at 2024-01-20 15:33:00+00:00.")
    lines.append("- Interpretation: p95=0 causes threshold saturation (every value >= 0), so these 'anomalies' are likely non-actionable without a different thresholding rule.")
    lines.append("")
else:
    lines.append("No log.error_count series present for candidate services in the window.")
    lines.append("")

# Row count findings
row_df = df[df['log_name'] == 'log.row_count']
if not row_df.empty:
    lines.append("log.row_count:")
    for _, r in row_df.iterrows():
        svc = r['cmdb_id']
        p95 = r['p95_threshold']
        tot = int(r['total_points_in_window'])
        anom = int(r['anomaly_count'])
        first_high = r['earliest_anomaly_utc']
        maxv = r['max_value_in_window']
        below_cnt = int(r.get('below_p15_count', 0))
        first_low = r.get('earliest_below_p15_utc', pd.NaT)
        lines.append(f"- {svc}: p95={p95}; total_points={tot}; high-anomalies={anom} (first at {first_high}; max={maxv}).")
        if below_cnt > 0:
            lines.append(f"  * {below_cnt} in-window values below global P15 (earliest at {first_low}) — indicates transient low row counts early in the window.")
    lines.append("")
else:
    lines.append("No log.row_count series present for candidate services in the window.")
    lines.append("")

# Short actionable interpretation
lines.append("Overall interpretation:")
lines.append("- The 'log.error_count' results are artifacts of a zero P95 threshold and should not be treated as meaningful alerts without adjusting thresholds or rule (e.g., use > P95 or require nonzero baseline).")
lines.append("- The 'log.row_count' series show both short high spikes (anomaly_count=2 for each service) and several in-window low-count occurrences below the global P15 at ~15:33 UTC for multiple services (currencyservice, checkoutservice, emailservice, recommendationservice).")
lines.append("- Recommendation: Investigate traffic/ingestion around ~15:33 UTC for the services with low row_count events and review currencyservice for both low counts and the later high spikes around ~15:41–15:45 UTC.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  p95_threshold  total_points_in_window  anomaly_count      earliest_anomaly_utc  max_value_in_window  below_p15_count    earliest_below_p15_utc
0        checkoutservice  log.error_count            0.0                      25             25 2024-01-20 15:33:00+00:00                    0                0                       NaT
1        currencyservice  log.error_count            0.0                      25             25 2024-01-20 15:33:00+00:00                    0                0                       NaT
2  recommendationservice  log.error_count            0.0                      25             25 2024-01-20 15:33:00+00:00                    0                0                       NaT
3           emailservice  log.error_count            0.0                      25             25 2024-01-20 15:33:00+00:00                    0                0                       NaT
4        currencyservice    log.row_count         2505.0                      25              2 2024-01-20 15:41:00+00:00                 2528                4 2024-01-20 15:33:00+00:00
5        checkoutservice    log.row_count          113.4                      25              2 2024-01-20 15:34:00+00:00                  117                3 2024-01-20 15:33:00+00:00
6           emailservice    log.row_count           37.8                      25              2 2024-01-20 15:34:00+00:00                   39                3 2024-01-20 15:33:00+00:00
7  recommendationservice    log.row_count          593.6                      25              2 2024-01-20 15:45:00+00:00                  612                4 2024-01-20 15:33:00+00:00```
```